<h1 style='font-weight:bold; color:#003366; text-align:center;'>Práctico 2 : Pandas, transformaciones y agrupaciones</h1>

<h3 style='font-weight:bold; color:#003366; text-align:center;'>Programación Analítica II</h3>
<h4 style='font-weight:bold; color:#003366; text-align:center;'>Docente Camila Palomeque</h4>
<h4 style='font-weight:bold; color:#003366; text-align:center;'>S2 . 2026</h4>

| | |
|---|---|
| **Temas evaluados** | Temas 4 y 5 |
| **Fecha de entrega** | Jueves 10 de septiembre, 23:59 |
| **Datasets** | `ventas_ecommerce.csv` y `indicadores_macro_uy.csv` |

**La tabla no es el resultado, es la evidencia.** Cada ejercicio
pide el cálculo *y* qué dice ese cálculo sobre el negocio. La interpretación vale puntos.

<h2 style='font-weight:bold; color:#003366;'>Instrucciones de entrega</h2>

1. Nombrá el archivo `Practico2_ApellidoNombre.ipynb`, con tu apellido y tu nombre.
2. Subilo a la tarea correspondiente del Moodle antes de la fecha límite.
3. **Ejecutá todas las celdas de arriba abajo antes de entregar** y verificá que no haya errores.
   Un notebook que no corre pierde puntos, sin importar qué diga el código.
   En Jupyter: *Kernel → Restart Kernel and Run All Cells*.
4. Las interpretaciones van en celdas de **texto** (markdown), no en comentarios del código.
5. El código tiene que cumplir con lo visto en los temas 1 y 2: nombres descriptivos, funciones
   cortas, docstrings de una línea, constantes con nombre en lugar de números sueltos y estilo PEP 8.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)


def ruta_datos(nombre_archivo):
    """Busca el archivo en las ubicaciones habituales del curso, incluida Colab."""
    candidatas = [
        Path("../Datos") / nombre_archivo,
        Path("Datos") / nombre_archivo,
        Path("/content/Datos") / nombre_archivo,
        Path(nombre_archivo),
    ]
    for candidata in candidatas:
        if candidata.exists():
            return str(candidata)
    raise FileNotFoundError(f"No se encontró {nombre_archivo}.")


# Todo lo que el notebook exporte va acá y no al lado del notebook.
CARPETA_SALIDAS = Path("salidas")
CARPETA_SALIDAS.mkdir(exist_ok=True)

print("pandas", pd.__version__)
print("Archivo de datos:", ruta_datos("ventas_ecommerce.csv"))

pandas 2.3.3
Archivo de datos: ventas_ecommerce.csv


<div style="border-left: 5px solid #003366; padding: 12px 16px; background-color: #F0F4F8;">
<b style="color:#003366;">El contexto</b><br><br>
Trabajás en el área de Administración y Finanzas de un comercio electrónico uruguayo. La gerencia
comercial pide dos cosas para la reunión del mes: <b>saber de dónde sale el margen</b> y
<b>saber si el negocio está creciendo o solamente rebotando con la estacionalidad</b>.<br><br>
El archivo <code>ventas_ecommerce.csv</code> es lo que sale del sistema: una fila por línea de orden,
de enero de 2024 a junio de 2026. El supuesto de margen bruto lo da el área comercial:
<b>35% de la facturación antes de logística</b>.
</div>

---

<h2 style='font-weight:bold; color:#003366;'>Parte 1 : carga y transformación <span style='color:#BB3300; font-size:0.8em;'>(20 puntos)</span></h2>

<h3 style='font-weight:bold; color:#0033BB;'>Ejercicio 1.1* <span style='color:#BB3300; font-size:0.8em;'>(10 pts)</span></h3>

Cargá `ventas_ecommerce.csv` con los tipos de datos correctos (fechas cómo fechas, categoricas como str u objects, enteros cómo int y continuos como float)

Después hacé el **análisis exploratorio básico** del tema 4: dimensiones, tipos, nulos, rango de fechas
y `describe()`.

Interpretación, en markdown, cuatro puntos cortos:

1. **¿Qué es una fila en esta tabla?** 
2. El rango de fechas que cubre. **¿Está completo el último mes? ¿Y el último año?**
3. Dos comentarios de los valores del `describe()` que valga la pena mirar dos veces. Ej. Hay al menos un mínimo que no debería poder existir: encontralo y decí qué harías con él.

In [2]:
# Tu código acá
#Carga de datos con tipos especificados
ventas = pd.read_csv(
    ruta_datos("ventas_ecommerce.csv"),
    parse_dates=["fecha"],
    dtype={
        "id_orden": str,
        "categoria": "category",
        "producto": str,
        "region": "category",
        "canal": "category",
        "unidades": int,
        "precio_unitario": float,
        "descuento": float,
        "costo_envio": float,
        "cliente_nuevo": bool,
    },
)

print("1. Tipos de dato")
print(ventas.dtypes)

print("\n2. Faltantes por columna")
print(ventas.isna().sum().loc[lambda serie: serie > 0])

print("\n3. Rango de fechas:", ventas["fecha"].min().date(), "a", ventas["fecha"].max().date())

print("\n4. Escala de las columnas numéricas")
display(ventas[["unidades", "precio_unitario", "descuento", "costo_envio"]].describe().round(2))

print("\n4. Distribución de las columnas categóricas")
display(ventas.describe(include=["category", "object"]).round(2))

1. Tipos de dato
id_orden                   object
fecha              datetime64[ns]
categoria                category
producto                   object
region                   category
canal                    category
unidades                    int64
precio_unitario           float64
descuento                 float64
costo_envio               float64
cliente_nuevo                bool
dtype: object

2. Faltantes por columna
Series([], dtype: int64)

3. Rango de fechas: 2024-01-01 a 2026-06-30

4. Escala de las columnas numéricas


,unidades,precio_unitario,descuento,costo_envio
count,8637.00,8637.00,8637.00,8637.00
mean,2.48,6688.14,0.08,303.04
std,1.12,6703.11,0.09,106.32
min,1.00,392.44,0.00,-36.29
25%,1.00,2269.85,0.00,221.91
50%,2.00,4074.07,0.10,294.37
75%,3.00,6303.40,0.15,378.34
max,4.00,28847.46,0.25,688.44



4. Distribución de las columnas categóricas


,id_orden,categoria,producto,region,canal
count,8637,8637,8637,8637,8637
unique,8637,5,19,6,4
top,ORD-0000-00000,Indumentaria,Zapatillas,Montevideo,Web
freq,1,2326,598,3267,3554


_(tu interpretación acá)_
1. **¿Qué es una fila en esta tabla?** 
- Cada fila representa una orden de compra realizada por un cliente en el e-commerce (una transacción de compra individual identificada por id_orden).

2. El rango de fechas que cubre. **¿Está completo el último mes? ¿Y el último año?**
- La tabla cubre desde el 01/01/2024 hasta el 30/06/2026 (8637 registros en total sin nulos).
Último mes (junio de 2026): Cierra en la fecha 2026-06-30, por lo que el mes de junio sí está completo.
Último año (2026): Está incompleto, solo toma el primer semestre (enero a junio).

3. Dos comentarios de los valores del `describe()` que valga la pena mirar dos veces. Ej. Hay al menos un mínimo que no debería poder existir: encontralo y decí qué harías con él.
- Flete negativo: La columna costo_envio tiene un valor mínimo de $-36.29. Esto es un error de carga o un reembolso de envío mal categorizado. Se podría mejorar cambiandolo por el costo promedio de un envío a esa región, o sino directamente cambiando su signo a positivo.
- Dispersión en el precio unitario: El precio promedio es de $6,688.14, pero la mediana ($4,074.07) y el percentil 75% ($6,303.40) están muy por debajo del valor máximo ($28,847.46), lo que refleja una asimetría a la derecha que estaría siendo provocada por unos pocos productos de mucho valor.

<h3 style='font-weight:bold; color:#0033BB;'>Ejercicio 1.2* <span style='color:#BB3300; font-size:0.8em;'>(10 pts)</span></h3>

Crea las siguientes columnas nuevas:

| Columna | Qué es |
|---|---|
| `facturacion` | Lo que efectivamente se le cobró al cliente: unidades por precio unitario por uno menos el descuento |
| `margen_bruto` | Facturación por el supuesto de margen del área comercial |
| `margen_neto` | Margen bruto menos el costo de envío |
| `periodo` | El mes de la orden, como texto `AAAA-MM` |
| `tramo_ticket` | Cuartiles de facturación con etiquetas baja, media-baja, media, media-alta y alta. Puede hacerse con `qcut` (`pd.qcut(columna, numero_de_cortes, labels=[etiqueta1, ..., etiquetan])`) o definiendo los márgenes con `np.percentile` y un if) |

El supuesto de margen va como constante con nombre, arriba. 

Interpretación: informá cuántas órdenes tienen **margen neto negativo**, qué porcentaje del total son
y cuánto suman. Después contestá la pregunta que le interesa a la gerencia: **¿el problema es que
esas órdenes son chicas, o es que el flete es caro?** Respaldá con un número que separe las dos
explicaciones.

In [3]:
# Tu código acá
MARGEN_BRUTO_SUPUESTO = 0.35

# Las nuevas columnas
ventas["facturacion"] = ventas["unidades"] * ventas["precio_unitario"] * (1 - ventas["descuento"])

ventas["margen_bruto"] = ventas["facturacion"] * MARGEN_BRUTO_SUPUESTO

ventas["margen_neto"] = ventas["margen_bruto"] - ventas["costo_envio"]

ventas["periodo"] = ventas["fecha"].dt.to_period("M").astype(str)

ventas["tramo_ticket"] = pd.qcut(
    ventas["facturacion"],
    q=5,
    labels=["baja", "media-baja", "media", "media-alta", "alta"],
)

# Análisis de margen neto negativo
ordenes_negativas = ventas[ventas["margen_neto"] < 0]
cant_negativas = len(ordenes_negativas)
pct_negativas = (cant_negativas / len(ventas)) * 100
suma_negativas = ordenes_negativas["margen_neto"].sum()

print(f"Órdenes con margen neto negativo: {cant_negativas}")
print(f"Porcentaje sobre el total: {pct_negativas:.2f}%")
print(f"Suma total del margen neto negativo: ${suma_negativas:,.2f}")

print("\nFacturación promedio general:", ventas["facturacion"].mean().round(2))
print("Facturación promedio de órdenes negativas:", ordenes_negativas["facturacion"].mean().round(2))
print("Costo de envío promedio general:", ventas["costo_envio"].mean().round(2))
print("Costo de envío promedio de órdenes negativas:", ordenes_negativas["costo_envio"].mean().round(2))

Órdenes con margen neto negativo: 153
Porcentaje sobre el total: 1.77%
Suma total del margen neto negativo: $-16,844.84

Facturación promedio general: 15079.96
Facturación promedio de órdenes negativas: 809.19
Costo de envío promedio general: 303.04
Costo de envío promedio de órdenes negativas: 393.31


_(tu interpretación acá)_
Órdenes con margen neto negativo
- Hubieron 153 órdenes con margen neto negativo, que representan un 1.77% del total de transacciones, con una pérdida acumulada de $-16,844.84.

¿Órdenes chicas o flete caro?:
- El problema es que son órdenes muy chicas: 
- - La facturación promedio de las órdenes con margen negativo es de solo $809.19, mientras que la facturación promedio del general del e-commerce es de $15,079.96 (casi 19 veces mayor).
- - El costo de envío promedio de las órdenes negativas es de $393.31, más alto que el costo medio general de la plataforma ($303.04).

- Entonces éstas órdenes son muy chicas, y encima el costo de envío promedio de ellas es mayor que el promedio general.

---

<h2 style='font-weight:bold; color:#003366;'>Parte 2 : calidad de datos <span style='color:#BB3300; font-size:0.8em;'>(30 puntos)</span></h2>

`ventas_ecommerce.csv` viene limpio, que es la excepción y no la regla. Los archivos que
llegan de un sistema de gestión no se parecen a eso.

Para que esta parte tenga sobre qué trabajar, la celda de abajo **reproduce el export de la semana
pasada**, que es el que realmente te habrían mandado: el marketplace no informó el flete de parte de
sus órdenes y hubo filas repetidas porque el proceso de carga se reintentó.

In [4]:
# Celda dada: reproduce el export sucio del sistema. No la modifiques ni la borres.
# Es autónoma a propósito: no depende de cómo hayas llamado a tus tablas de la parte 1.
MARGEN_BRUTO_DEL_AREA_COMERCIAL = 0.35   # supuesto del área comercial, antes de logística

ventas_erp = (
    pd.read_csv(ruta_datos("ventas_ecommerce.csv"),
                parse_dates=["fecha"],
                dtype={"categoria": "category", "region": "category", "canal": "category"})
    .assign(
        facturacion=lambda tabla: (tabla["unidades"] * tabla["precio_unitario"]
                                   * (1 - tabla["descuento"])),
        margen_bruto=lambda tabla: tabla["facturacion"] * MARGEN_BRUTO_DEL_AREA_COMERCIAL,
    )
)

generador = np.random.default_rng(2026)
sin_reporte_de_flete = ((ventas_erp["canal"] == "Marketplace")
                        & (generador.random(len(ventas_erp)) < 0.30))
ventas_erp.loc[sin_reporte_de_flete, "costo_envio"] = np.nan
ventas_erp = pd.concat([ventas_erp, ventas_erp.sample(24, random_state=7)], ignore_index=True)
ventas_erp["margen_neto"] = ventas_erp["margen_bruto"] - ventas_erp["costo_envio"]

print("Export del ERP:", ventas_erp.shape)
ventas_erp.head(3)

Export del ERP: (8661, 14)


,id_orden,fecha,categoria,producto,region,canal,unidades,precio_unitario,descuento,costo_envio,cliente_nuevo,facturacion,margen_bruto,margen_neto
0,ORD-0000-00000,2024-01-01,Hogar,Silla ergonomica,Montevideo,Marketplace,1,5634.83,0.1,NaN,False,5071.347,1774.97145,NaN
1,ORD-0000-00001,2024-01-01,Indumentaria,Zapatillas,Colonia,Web,3,2315.23,0.0,173.71,True,6945.690,2430.99150,2257.2815
2,ORD-0000-00002,2024-01-01,Indumentaria,Pantalon,Montevideo,Marketplace,2,2495.15,0.1,233.84,False,4491.270,1571.94450,1338.1045


<h3 style='font-weight:bold; color:#0033BB;'>Ejercicio 2.1** <span style='color:#BB3300; font-size:0.8em;'>(15 pts)</span></h3>

Auditá `ventas_erp` y reportá:

1. Cuántos nulos hay por columna, en cantidad **y en porcentaje**.
2. Cuántas filas están 100% duplicadas y cuántos `id_orden` están repetidos. No son lo mismo:
   explicá la diferencia con los números de esta tabla.
3. **Si el faltante tiene patrón.** Calculá el porcentaje de `costo_envio` faltante agrupando por
   `canal` y por `region`, y decidí si falta al azar o no.

Interpretación: escribí en dos o tres oraciones qué le dirías al área de Sistemas. Y la pregunta que
importa: **si calculás el margen neto total ignorando los nulos, ¿el número que informás queda alto
o bajo, y por qué?**

In [5]:
# Tu código acá
# 1. Nulos en cantidad y porcentaje
nulos_cantidad = ventas_erp.isna().sum()
nulos_porcentaje = (ventas_erp.isna().mean() * 100).round(2)

print("1. AUDITORÍA DE NULOS")
for columna in ventas_erp.columns:
    cant = nulos_cantidad[columna]
    if cant > 0:
        pct = nulos_porcentaje[columna]
        print(f"Columna '{columna}': {cant} nulos ({pct}%)")

# 2. Duplicados exactos contra IDs repetidos
filas_duplicadas = ventas_erp.duplicated().sum()
id_orden_repetidos = ventas_erp["id_orden"].duplicated().sum()

print("\n2. AUDITORÍA DE DUPLICADOS")
print(f"Filas 100% duplicadas: {filas_duplicadas}")
print(f"IDs de orden repetidos: {id_orden_repetidos}")

# 3.
print("\n3. PORCENTAJE DE FALTANTES POR CANAL")
print((ventas_erp.groupby("canal", observed=False)["costo_envio"].apply(lambda x: x.isna().mean() * 100)).round(2))

print("\nPORCENTAJE DE FALTANTES POR REGIÓN")
print((ventas_erp.groupby("region", observed=False)["costo_envio"].apply(lambda x: x.isna().mean() * 100)).round(2))

1. AUDITORÍA DE NULOS
Columna 'costo_envio': 460 nulos (5.31%)
Columna 'margen_neto': 460 nulos (5.31%)

2. AUDITORÍA DE DUPLICADOS
Filas 100% duplicadas: 24
IDs de orden repetidos: 24

3. PORCENTAJE DE FALTANTES POR CANAL
canal
App movil       0.00
Marketplace    29.56
Telefono        0.00
Web             0.00
Name: costo_envio, dtype: float64

PORCENTAJE DE FALTANTES POR REGIÓN
region
Canelones         4.89
Colonia           4.76
Interior resto    6.06
Maldonado         6.38
Montevideo        5.01
Salto             5.31
Name: costo_envio, dtype: float64


_(tu interpretación acá)_

1 .
Nulos por columna:
- costo_envio: 460 nulos (5.31%)
- margen_neto: 460 nulos (5.31%) (derivados de la falta del costo de envío).

2 .
Duplicados:
- - Filas 100% duplicadas (24): Son registros donde todos los campos de la fila son idénticos a los de otra fila (mismo id_orden, misma fecha, mismo cliente, mismo producto, mismo monto, etc.).
- - id_orden repetidos (24): Indica cuántas veces aparece un identificador de orden que ya había sido registrado en la tabla.
- Diferencia: En este caso son lo mismo ya que ambas cifras coinciden exactamente (24 y 24) porque los duplicados detectados son clones exactos. No existen casos donde un mismo id_orden aparezca dos veces con datos distintos o contradictorios (por ejemplo, con un monto diferente o una fecha distinta). Las 24 repeticiones se deben únicamente a que 24 filas completas se exportaron o copiaron dos veces por error en el ERP.

3 .
Patrón de faltantes:
- - Por Canal: Marketplace tiene 29.56% de faltantes, mientras que App móvil, Teléfono y Web no tienen.
- - Por Región: El porcentaje de faltantes se distribuye homogéneamente entre todas las regiones (alrededor de entre 4.8% a 6.4%).
- Conclusión: No faltan al azar. Existe una falla en el canal de ventas Marketplace, donde casi el 30% de sus órdenes no están reportando el costo logístico.



Qué le diría al área de Sistemas:
- "Hay dos problemas en la extracción del ERP: las ventas que vienen de Marketplace están entrando sin el costo de envío en casi 1 de cada 3 casos (29,59% de las órdenes), y además se están añadiendo 24 filas de registros clonadas de más. Habría que revisar la integración que trae las ventas de Marketplace y la forma en que el sistema exporta la información para solucionar estos problemas."

¿El margen neto total informado queda alto o bajo si se ignoran los nulos?:
- El número informado queda bajo, porque al calcular el margen neto omitiendo las filas con costo_envio nulo, Pandas excluye por completo esas 460 transacciones del cálculo general. Al sacar esas órdenes, no solo dejamos de restar su costo de envío, sino que también dejamos de sumar toda la facturación y el margen bruto que esas ventas generaron. Como el margen bruto promedio de esas compras supera en promedio al costo de flete, omitirlas estaría reduciendo el margen neto total informado.

<h3 style='font-weight:bold; color:#0033BB;'>Ejercicio 2.2** <span style='color:#BB3300; font-size:0.8em;'>(15 pts)</span></h3>

Ahora decidí qué hacer, y dejá el rastro de la decisión para los **duplicados** y para los **nulos de `costo_envio`**, mencioná en la explicación:

1. Qué se encontró
2. Cuántos (filas y %)
3. Cuál es la decisión de tratamiento
4. Por qué 

Aplicá las dos decisiones y **medí el impacto**: cuántas filas quedaron, y cuánto se movió la
facturación total y el margen neto total antes y después. El impacto tiene que ser un número
calculado en el notebook, no una frase.

Una de las dos decisiones puede perfectamente ser **no hacer nada**, si la justificás.

In [6]:
# Tu código acá
# Lo inicial
filas_ini = len(ventas_erp)
fact_ini = ventas_erp["facturacion"].sum()
m_neto_ini = ventas_erp["margen_neto"].sum()

# Eliminar las filas clonadas
ventas_limpio = ventas_erp.drop_duplicates().copy()

# Imputar el costo_envio con el promedio del canal Marketplace
flete_prom_mkt = ventas_limpio.loc[
    ventas_limpio["canal"] == "Marketplace", "costo_envio"
].mean()

ventas_limpio["costo_envio"] = ventas_limpio["costo_envio"].fillna(
    flete_prom_mkt
)

# Recalcular margen neto con el flete imputado
ventas_limpio["margen_neto"] = (
    ventas_limpio["margen_bruto"] - ventas_limpio["costo_envio"]
)

# Estado final
filas_fin = len(ventas_limpio)
fact_fin = ventas_limpio["facturacion"].sum()
m_neto_fin = ventas_limpio["margen_neto"].sum()

# Medición del impacto
print("MEDICIÓN DE IMPACTO")
print(f"Filas: {filas_ini} -> {filas_fin} (Diferencia: {filas_fin - filas_ini})")
print(
    f"Facturación Total: ${fact_ini:,.2f} -> ${fact_fin:,.2f} (Variación:"
    f" ${fact_fin - fact_ini:,.2f})"
)
print(
    f"Margen Neto Total: ${m_neto_ini:,.2f} -> ${m_neto_fin:,.2f} (Variación:"
    f" ${m_neto_fin - m_neto_ini:,.2f})"
)

MEDICIÓN DE IMPACTO
Filas: 8661 -> 8637 (Diferencia: -24)
Facturación Total: $130,428,121.74 -> $130,245,617.13 (Variación: $-182,504.61)
Margen Neto Total: $40,804,136.83 -> $42,969,836.02 (Variación: $2,165,699.19)


_(tu tabla acá)_

Tratamiento de Duplicados
- Qué se encontró y cuántos: 24 filas 100% duplicadas.
- Decisión: Eliminar las filas duplicadas mediante .drop_duplicates().
- Por qué: Son registros idénticos clonados por un error en la exportación del ERP. Mantenerlos duplicaría artificialmente la facturación y el margen de esas ventas.

Tratamiento de Nulos de costo_envio
- Qué se encontró y cuántos: 460 valores nulos (5.31% del total), concentrados en el canal Marketplace.
- Decisión: Reemplazar los nulos con la media del costo de envío del canal Marketplace.
- Por qué: Las ventas de Marketplace sí pasaron y generaron margen bruto; eliminar las filas descartaría transacciones reales. Imputar con el costo logístico promedio de ese mismo canal asigna un costo estimado realista sin distorsionar el margen del negocio.

Impacto Medido
- Filas: Pasaron de 8.661 a 8.637 (-24 filas duplicadas).
- Facturación total: Bajó de $130,428,121.74 a $130,245,617.13 (-$182,504.61), corrigiendo la sobreestimación por ventas repetidas.
- Margen Neto total: Subió de $40,804,136.83 a $42,969,836.02 (+$2,165,699.19), reflejando la ganancia de las 460 ventas de Marketplace al incorporarles su costo estimado.

---

<h2 style='font-weight:bold; color:#003366;'>Parte 3 : agrupaciones y transformaciones <span style='color:#BB3300; font-size:0.8em;'>(20 puntos)</span></h2>

De acá en adelante volvé a trabajar con la tabla **limpia** de la parte 1, no con el export
del ERP.

<h3 style='font-weight:bold; color:#0033BB;'>Ejercicio 3.1 <span style='color:#BB3300; font-size:0.8em;'>(10 pts)</span></h3>

Armá una tabla resumen **por categoría** con agregaciones con nombre, que
tenga como mínimo: órdenes, unidades, facturación total, ticket promedio, ticket mediano, descuento
promedio y participación de cada categoría en la facturación total.

Agregá una columna más: la **brecha entre el promedio y la mediana** del ticket, en porcentaje.
Ordená la tabla por facturación.

Interpretación, y es la parte que más pesa de este ejercicio: **¿qué te dice esa brecha?** Contestá
para las dos categorías con la brecha más grande y la más chica. Y decidí, con argumento: si tuvieras
que informarle a la gerencia **un solo** número de ticket por categoría, ¿el promedio o la mediana?

In [7]:
# Tu código acá
resumen_categoria = (
    ventas.groupby("categoria", observed=False)
    .agg(
        ordenes=("id_orden", "count"),
        unidades=("unidades", "sum"),
        facturacion_total=("facturacion", "sum"),
        ticket_promedio=("facturacion", "mean"),
        ticket_mediano=("facturacion", "median"),
        descuento_promedio=("descuento", lambda x: (x.mean() * 100)),
    )
    .assign(
        participacion_facturacion=lambda df: (
            df["facturacion_total"] / df["facturacion_total"].sum() * 100
        ),
        brecha_promedio_mediana=lambda df: (
            (df["ticket_promedio"] - df["ticket_mediano"])
            / df["ticket_mediano"]
            * 100
        ),
    )
    .sort_values(by="facturacion_total", ascending=False)
)

display(
    resumen_categoria.style.format({
        "facturacion_total": "${:,.2f}",
        "ticket_promedio": "${:,.2f}",
        "ticket_mediano": "${:,.2f}",
        "descuento_promedio": "{:.2f}%",
        "participacion_facturacion": "{:.2f}%",
        "brecha_promedio_mediana": "{:.2f}%",
    })
)

,ordenes,unidades,facturacion_total,ticket_promedio,ticket_mediano,descuento_promedio,participacion_facturacion,brecha_promedio_mediana
categoria,,,,,,,,
Electronica,1952,4748,"$80,541,887.58","$41,261.21","$38,701.92",8.06%,61.84%,6.61%
Hogar,1783,4386,"$20,857,983.16","$11,698.25","$11,209.40",8.56%,16.01%,4.36%
Deportes,1516,3828,"$14,428,583.41","$9,517.54","$9,082.91",8.29%,11.08%,4.79%
Indumentaria,2326,5807,"$12,258,361.23","$5,270.15","$5,049.09",8.56%,9.41%,4.38%
Libros,1060,2609,"$2,158,801.75","$2,036.61","$1,964.12",8.22%,1.66%,3.69%


_(tu interpretación acá)_
La brecha entre el promedio y la mediana mide el sesgo o asimetría en la distribución de las ventas de cada categoría. Una brecha positiva significa que el ticket promedio está inflado por un grupo chico de compras atípicamente altas, moviendo la media por encima del valor representativo típico.

Análisis de los extremos:
- Mayor brecha -> Electrónica (6.61%): Presenta la mayor asimetría. El ticket promedio ($41,261.21) es mayor al ticket mediano ($38,701.92), lo que indica la presencia de compras de alto valor que empujan el promedio hacia arriba.
- Menor brecha -> Libros (3.69%): Es la categoría con los tickets más estandarizados. La diferencia entre lo que se cobra en promedio ($2,036.61) y el punto medio exacto ($1,964.12) es muy chica.

Métrica que recomiendo: La Mediana.
- Porque el ticket mediano refleja la compra real del cliente típico o central (el 50% de las órdenes está por debajo y el 50% por encima) sin verse distorsionado por transacciones extraordinarias. Informar solo el promedio generaría una falsa expectativa de ingreso por orden, principalmente en categorías pesadas como Electrónica donde las ventas atípicas distorsionan la media.

<h3 style='font-weight:bold; color:#0033BB;'>Ejercicio 3.2 <span style='color:#BB3300; font-size:0.8em;'>(10 pts)</span></h3>

Con `transform`, agregá a la tabla de ventas dos columnas:

- `facturacion_categoria`: la facturación total de la categoría de cada orden, repetida en cada fila.
- `participacion_en_categoria_pct`: qué porcentaje de la facturación de su categoría explica cada
  orden.

Después mostrá las diez órdenes con mayor participación en su categoría, y calculá **cuánto de la
facturación de cada categoría explican sus diez órdenes más grandes**.

Interpretación: ¿en qué categoría está más concentrada la facturación? Y la consecuencia: si el mes
que viene se caen las tres órdenes más grandes de esa categoría, ¿cuánto se pierde en pesos?

In [8]:
# Tu código acá
ventas["facturacion_categoria"] = ventas.groupby("categoria", observed=False)["facturacion"].transform("sum")

ventas["participacion_en_categoria_pct"] = (
    ventas["facturacion"] / ventas["facturacion_categoria"]
) * 100

top_10_ordenes_participacion = ventas.sort_values(
    by="participacion_en_categoria_pct", ascending=False
).head(10)[
    [
        "id_orden",
        "categoria",
        "producto",
        "facturacion",
        "participacion_en_categoria_pct",
    ]
]

print("Top 10 órdenes con mayor participación en su categoría:")
display(top_10_ordenes_participacion.round(4))

# Porcentaje de facturación que explican las 10 órdenes más grandes de cada categoría
top10_facturacion_por_cat = (
    ventas.groupby("categoria", observed=False)["facturacion"]
    .apply(lambda s: s.nlargest(10).sum())
    .reset_index(name="facturacion_top10")
)

totales_por_cat = (
    ventas.groupby("categoria", observed=False)["facturacion"]
    .sum()
    .reset_index(name="facturacion_total")
)

concentracion_categoria = pd.merge(
    top10_facturacion_por_cat, totales_por_cat, on="categoria"
).assign(
    pct_explicado_top10=lambda df: (df["facturacion_top10"] / df["facturacion_total"]) * 100
).sort_values(by="pct_explicado_top10", ascending=False)

print("\nPorcentaje de la facturación explicado por las 10 órdenes más grandes de cada categoría:")
display(
    concentracion_categoria.style.format({
        "facturacion_top10": "${:,.2f}",
        "facturacion_total": "${:,.2f}",
        "pct_explicado_top10": "{:.2f}%",
    })
)

# Impacto económico al caer las 3 órdenes más grandes de la categoría más concentrada
cat_mas_concentrada = concentracion_categoria.iloc[0]["categoria"]
top3_ordenes_cat = ventas[ventas["categoria"] == cat_mas_concentrada].nlargest(3, "facturacion")
perdida_3_ordenes = top3_ordenes_cat["facturacion"].sum()

print(f"\nCategoría con mayor concentración: {cat_mas_concentrada}")
print(f"Pérdida en pesos si se caen las 3 órdenes más grandes: ${perdida_3_ordenes:,.2f}")

Top 10 órdenes con mayor participación en su categoría:


,id_orden,categoria,producto,facturacion,participacion_en_categoria_pct
463,ORD-0044-00463,Libros,Clean Code,5023.72,0.2327
5298,ORD-0540-05298,Libros,Clean Code,4842.24,0.2243
680,ORD-0064-00680,Libros,Estadistica aplicada,4819.56,0.2233
3469,ORD-0375-03469,Libros,Clean Code,4672.24,0.2164
3067,ORD-0341-03067,Libros,Clean Code,4578.84,0.2121
6743,ORD-0736-06743,Libros,Python para finanzas,4568.84,0.2116
8330,ORD-0881-08330,Libros,Clean Code,4508.72,0.2089
6014,ORD-0643-06014,Libros,Python para finanzas,4441.88,0.2058
4626,ORD-0476-04626,Libros,Estadistica aplicada,4430.44,0.2052
4039,ORD-0428-04039,Libros,Estadistica aplicada,4425.40,0.2050



Porcentaje de la facturación explicado por las 10 órdenes más grandes de cada categoría:


,categoria,facturacion_top10,facturacion_total,pct_explicado_top10
4,Libros,"$46,311.88","$2,158,801.75",2.15%
0,Deportes,"$222,615.84","$14,428,583.41",1.54%
2,Hogar,"$288,902.48","$20,857,983.16",1.39%
1,Electronica,"$1,015,698.92","$80,541,887.58",1.26%
3,Indumentaria,"$127,167.32","$12,258,361.23",1.04%



Categoría con mayor concentración: Libros
Pérdida en pesos si se caen las 3 órdenes más grandes: $14,685.52


_(tu interpretación acá)_

Categoría más concentrada: 
- La categoría con mayor nivel de concentración es Libros, donde sus 10 órdenes más grandes representan el 2.15% del total facturado por la categoría ($46,311.88 de un total de $2,158,801.75).
- En general, el e-commerce presenta una baja dependencia de órdenes individuales. En la categoría menos concentrada (Indumentaria), las 10 órdenes principales explican solo el 1.04% de la facturación.

Si el mes que viene se cayeran las tres órdenes más grandes de la categoría Libros, la pérdida total ascendería a $14,685.52.

---

<h2 style='font-weight:bold; color:#003366;'>Parte 4 : evolución y uniones <span style='color:#BB3300; font-size:0.8em;'>(30 puntos)</span></h2>

<h3 style='font-weight:bold; color:#0033BB;'>Ejercicio 4.1*** <span style='color:#BB3300; font-size:0.8em;'>(20 pts)</span></h3>

Dada la serie de **facturación mensual** agregale tres columnas:
variación respecto del mes anterior, **variación interanual** (ver pct_change() para ambas variaciones)y media móvil de tres meses (ver cómo se usa rolling()).

Interpretación, y es lo que la gerencia quiere saber:

1. ¿El negocio está creciendo? Contestá con la variación interanual, no con la mensual, y explicá en
   una oración por qué esa es la comparación correcta en un negocio con estacionalidad.
2. ¿Cuál fue el mejor mes de la serie y por qué? ¿Es una buena noticia?
3. El último mes de la serie: ¿lo usarías para la conclusión? Justificá.

In [9]:
# Tu código acá

# Serie mensual de facturación
serie_mensual = (
    ventas.set_index("fecha")["facturacion"]
    .resample("ME")
    .sum()
    .to_frame(name="facturacion")
)

serie_mensual["var_mensual_pct"] = serie_mensual["facturacion"].pct_change() * 100
serie_mensual["var_interanual_pct"] = serie_mensual["facturacion"].pct_change(12) * 100
serie_mensual["media_movil_3m"] = serie_mensual["facturacion"].rolling(window=3).mean()

serie_mensual_display = serie_mensual.copy()
serie_mensual_display.index = serie_mensual_display.index.strftime("%Y-%m")

display(
    serie_mensual_display.style.format({
        "facturacion": "${:,.2f}",
        "var_mensual_pct": "{:+.2f}%",
        "var_interanual_pct": "{:+.2f}%",
        "media_movil_3m": "${:,.2f}",
    }, na_rep="-")
)

,facturacion,var_mensual_pct,var_interanual_pct,media_movil_3m
fecha,,,,
2024-01,"$4,686,956.32",-,-,-
2024-02,"$4,582,703.22",-2.22%,-,-
2024-03,"$5,915,739.58",+29.09%,-,"$5,061,799.71"
2024-04,"$5,849,118.61",-1.13%,-,"$5,449,187.14"
2024-05,"$5,757,655.54",-1.56%,-,"$5,840,837.91"
2024-06,"$4,372,742.70",-24.05%,-,"$5,326,505.62"
2024-07,"$3,488,796.88",-20.21%,-,"$4,539,731.71"
2024-08,"$3,242,851.26",-7.05%,-,"$3,701,463.61"
2024-09,"$2,767,882.09",-14.65%,-,"$3,166,510.07"


_(tu interpretación acá)_

1. ¿El negocio está creciendo?
- El negocio no presenta un crecimiento estructural sostenible; se encuentra estancado con cambios cíclicos. La variación interanual oscila permanentemente entre caídas fuertes (ej. -23.72% en agosto 2025 o -20.52% en abril 2026) y repuntes moderados (ej. +11.18% en octubre 2025).
- Justificación de la métrica: La variación interanual es la métrica adecuada porque elimina el efecto de la estacionalidad (comparando directamente el mismo mes entre distintos años), permitiendo aislar la tendencia real del negocio frente a caídas o subas naturales de la demanda.

2. ¿Cuál fue el mejor mes de la serie y por qué? ¿Es una buena noticia?
- El mejor mes fue abril de 2025, alcanzando el pico máximo de facturación de la serie con $6,068,654.51 (+3.75% interanual).
- No es necesariamente una buena noticia aislada, ya que no respondió a un salto de escala sostenido del negocio. La media móvil de 3 meses siguiente se mantuvo cayendo hacia la segunda mitad del año, confirmando que se trató únicamente del pico estacional de la primera mitad del año.

3. El último mes de la serie: ¿lo usarías para la conclusión?
- A junio de 2026 no lo usaría como base para concluir sobre la salud general del negocio ya que el mes cierra con una facturación de $4,240,472.73 (-1.06% interanual). Al tratarse del cierre del primer semestre, evaluar el desempeño general tomando únicamente un mes individual (que además coincide con la entrada a la temporada baja de invierno) no sería útil para ver como va en general la empresa.

<h3 style='font-weight:bold; color:#0033BB;'>Ejercicio 4.2*** <span style='color:#BB3300; font-size:0.8em;'>(10 pts)</span></h3>

<code>indicadores_macro_uy.csv</code> tiene la serie mensual de
actividad, inflación, tipo de cambio y salario real. 

Realiza un <code>merge</code> por mes contra tu serie de facturación y evalua si la caída de un mes fue del negocio o del contexto.<br><br>
</div>

In [11]:
# 1. Cargar datos macro y llevar la fecha a período mensual
macro = pd.read_csv('indicadores_macro_uy.csv')
macro['fecha'] = pd.to_datetime(macro['fecha'])
macro['periodo'] = macro['fecha'].dt.to_period('M')

df_ventas = serie_mensual.copy()
df_ventas['periodo'] = df_ventas.index.to_period('M')

df_merge = pd.merge(df_ventas, macro, on='periodo', how='inner')

df_merge['var_actividad_interanual_pct'] = df_merge['indice_actividad'].pct_change(12) * 100

meses_caida_ventas = df_merge[df_merge['var_interanual_pct'] < 0].copy()

columnas_ver = [
    'periodo', 
    'var_interanual_pct', 
    'var_actividad_interanual_pct', 
    'salario_real_indice',
    'desempleo'
]

print("Análisis de meses con caída interanual de ventas:")
display(meses_caida_ventas[columnas_ver].dropna().round(2))

Análisis de meses con caída interanual de ventas:


,periodo,var_interanual_pct,var_actividad_interanual_pct,salario_real_indice,desempleo
12,2025-01,-9.42,-1.26,113.02,8.94
14,2025-03,-1.66,3.77,111.86,8.72
16,2025-05,-15.62,2.26,113.70,8.56
17,2025-06,-1.99,2.23,112.70,8.90
18,2025-07,-4.52,1.88,113.40,8.33
19,2025-08,-23.72,1.69,114.08,8.71
22,2025-11,-8.23,3.85,114.07,8.40
23,2025-12,-5.73,2.48,115.65,7.91
24,2026-01,-1.48,4.10,115.24,7.21
27,2026-04,-20.52,4.88,114.77,7.94


Las caídas en la facturación fueron por problemas internos del negocio, no del contexto macroeconómico.

Esto es porque mirando la tabla, en los meses con las peores caídas de ventas (como agosto 2025 con -23.72% o abril 2026 con -20.52%), la actividad económica del país estaba creciendo (entre +1.6% y +6%).

Además, durante esas caídas del negocio, el desempleo venía bajando y el salario real se mantenía estable o al alza. Es decir, la empresa perdió ventas en momentos donde la economía mejoraba y la gente tenía mayor poder adquisitivo, lo que descarta que la culpa sea de una recesión externa.